# Getting started with MemsArrayH5 objects

The `MemsArrayH5` class allows getting signals from MemsArray saved in a H5 file with data formated as MuH5 format

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros_tools.log import log
from megamicros.core.h5 import MemsArrayH5

log.setLevel( "INFO" )
# log.setLevel("WARNING")

# Choose a file where some H5 files are stored
DIRECTORY = '/Users/brunogas/Documents/Data/'

# Choose a H5 file
FILENAME = DIRECTORY + 'mu5h-20220816-051126.h5'

## Starting with simple signals

One has only to specify the file or directory name where to find H5 files.

The ``MemsArrayH5`` open the input H5 file or the first H5 file of the directory list and populates the antenna parameters with metadata read from the file. An exception is raised if file or directory are not found.

In [ ]:
# Define the antenna
antenna = MemsArrayH5( 
    filename=FILENAME
)

You can get now some Meta informations regarding the file

In [ ]:
print( f"Sampling frequency: {antenna.sampling_frequency}Hz" )
print( f"Available MEMs number: {antenna.available_mems_number}" )
print( f"Whether counter is available or not: {antenna.counter}" )

## Getting and plotting some MEMs signals

You can select some MEMs you would like to plot and get signals on a given range time (10 seconds in this example)

In [ ]:
# Run antenna
antenna.run(
    mems = [3, 4],
    duration=12,
    counter_skip = True,
    datatype='int32'
)

# Init a np.ndarray
signals = np.ndarray( (0, antenna.channels_number ) )

# Get signals
for data in antenna:
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
antenna.wait()
print( f"exit from loop. Signal shape is: {np.shape( signals )}" )

Here is the plotting program

In [ ]:
# plot signals
time = np.array( range( np.size(signals,0) ) )/antenna.sampling_frequency
#time = np.array( range( 1000 ) )/antenna.sampling_frequency
fig, axs = plt.subplots( antenna.channels_number )
fig.suptitle('Channels activity')	
for s in range( antenna.channels_number ):
    axs[s].plot( time, signals[:len(time),s] )
    axs[s].set( xlabel='time in seconds', ylabel='channel %d' % s )

plt.show()

## Saving signals as wave file

In [ ]:
import wave

WAV_FILENAME = 'titi.wav'

# 2 seconds run, getting signals from MEMs 1 and 2
antenna.run(
    mems = [5, 6],
    duration=5,
    counter_skip = True,
    datatype='int32'
)

with  wave.open( WAV_FILENAME, mode='wb' ) as wavfile:
    wavfile.setnchannels(2)
    wavfile.setsampwidth(2)
    wavfile.setframerate( antenna.sampling_frequency )

    # Get signals
    for data in antenna:
        signal = data >> 4
        wavfile.writeframesraw( np.int16( np.reshape( signal, np.size( signal ), order='C' ) ) )

# waiting for the end of the running thread is mandatory
antenna.wait()

## Hearing signal with *pyaudio* library

In [ ]:
import pyaudio

FRAME_LENGTH = 512

# Instantiate PyAudio and initialize PortAudio system resources (1)
p = pyaudio.PyAudio()

# Open stream
stream = p.open(
    format = pyaudio.paFloat32,
    channels = 2,
    rate = int( antenna.sampling_frequency ),
    output=True,
    frames_per_buffer=FRAME_LENGTH,
)

# Start running the remote Megamicros system
antenna.run( 
    mems=[1, 2],
    duration=10,
    frame_length=FRAME_LENGTH,
    counter_skip = True,
    datatype='int32',
    signal_q_size = 0,
)

# Get signals
transfers_counter = 0
for data in antenna:
    signal = data >> 4

    # convert into float and normalize with MEMs sensibility
    data = ( data.astype( np.float32 ) * antenna.sensibility )

    # write into audio stream
    stream.write( data.tobytes( order='C'), num_frames=FRAME_LENGTH )
    transfers_counter += 1

# Close stream and release PortAudio system resources (5)
stream.close()            
p.terminate()
antenna.wait()

# Saving signals

In the following, *real time* is no longer necessary, unlike in the previous examples.
By stopping the real time you can load signals faster. 
You have only to set the `real_time` option to `False`:

In [ ]:

# Run antenna to download the file entirely
antenna.run(
    mems = [6],
    duration=400,
    counter_skip = True,
    datatype='int32',
    real_time=False,
    start_time=150
)

# Init a np.ndarray
signals2 = np.ndarray((0, antenna.channels_number))

# Get signals
for data in antenna:
    signals2 = np.concatenate((signals2, data), axis=0)
    
# waiting for the end of the running thread is mandatory
antenna.wait()

In [ ]:
import IPython.display

# plot signals
time_axis = np.array(range(np.size(signals2,0)))/antenna.sampling_frequency
fig, axs = plt.subplots(antenna.channels_number)
fig.suptitle('Channels activity')

for s in range(antenna.channels_number):
    if antenna.channels_number > 1:
        axs[s].plot(time_axis/60, signals2[:len(time_axis), s])
        axs[s].set(xlabel='time in minutes', ylabel=f'channel {s}')
    else:
        axs.plot(time_axis/60, signals2[:len(time_axis), s])
        axs.set(xlabel='time in minutes', ylabel=f'channel {s}')

plt.show()

IPython.display.Audio(signals2[:,0], rate=antenna.sampling_frequency)

# Getting MµH5 annotations

MµH5 files can also embed annotations. You can access them by using the `annotations` method:

In [8]:
# Choose a H5 file in your H5 files directory
from pprint import pprint
DIRECTORY = '/Users/brunogas/Documents/Dev/Bimea/aidb/data/muh5/'
FILENAME = DIRECTORY + 'mu5h-20220816-051126-annotated.h5'
# mu5h-20220715-022743-annotated.h5 -> no anotations
# mu5h-20220816-051126-annotated.h5

# Define the antenna
antenna = MemsArrayH5( 
    filename=FILENAME
)

# Get annotations
annotations = antenna.annotations

if annotations is None or len( annotations ) == 0:
    print( f"No annotations in file {FILENAME}" )
else:
    pprint( f"annotations: {annotations}")

# Get signal
signal = antenna.get(channels=(0,) )
print( f"signal shape: {np.shape( signal )}" )

# plot annotations with bar plot
count = len( annotations['sample_start'] )
x = []
height = [1 for i in range( count ) ]
width = []
labels = []
for i in range( count ):
    x.append( ( annotations['sample_end'][i] - annotations['sample_start'][i] )/2 + annotations['sample_start'][i] )
    width.append( annotations['sample_end'][i] - annotations['sample_start'][i] )
    labels.append( str( annotations['label_id'][i] ) )

# Add a last bar to cover all signal for plotting purpose
x.append( np.size( signal, 1 ) ) 
width.append( 1 )
labels.append( '' )
height.append( 1 )

fig, axs = plt.subplots( 2 )
fig.suptitle('Channel signal and annotations')	

time = np.array( range( np.size(signal,1) ) )/antenna.sampling_frequency
axs[0].plot( time, signal[0,:])
axs[0].set( xlabel='time in seconds', ylabel=f'channel {0}' )
axs[1].bar( x, height=height, width=width, align='center' )
axs[1].set( xlabel='samples', ylabel='annotations', title='Annotations' )


2024-02-19 17:42:57,165 [INFO]:  .Install MemsArrayH5 settings
2024-02-19 17:42:57,167 [INFO]:  .Created a new antenna
2024-02-19 17:42:57,169 [INFO]:  .Found /Users/brunogas/Documents/Dev/Bimea/aidb/data/muh5/mu5h-20220715-022743-annotated.h5 MuH5 file
2024-02-19 17:42:57,171 [INFO]:  .Set 2 available MEMs numbered from 0 to 1
2024-02-19 17:42:57,171 [INFO]:  .No analogic channels available
2024-02-19 17:42:57,172 [INFO]: found MuH5 group
2024-02-19 17:42:57,173 [INFO]:  .Processing /Users/brunogas/Documents/Dev/Bimea/aidb/data/muh5/mu5h-20220715-022743-annotated.h5 H5 file... 
2024-02-19 17:42:57,175 [INFO]:  .900s (15.00min) of data in H5 file
2024-02-19 17:42:57,175 [INFO]:  .Whether counter is available: True
2024-02-19 17:42:57,176 [INFO]:  .2 available mems
2024-02-19 17:42:57,176 [INFO]:  .0 available analogic channels
2024-02-19 17:42:57,176 [INFO]:  .Requested channels: (0,)
2024-02-19 17:42:57,177 [INFO]:  .Total available channels number is 3
2024-02-19 17:42:57,177 [INFO]:

'annotations: {}'
No annotations in file /Users/brunogas/Documents/Dev/Bimea/aidb/data/muh5/mu5h-20220715-022743-annotated.h5
signal shape: (1, 9000000)


KeyError: 'sample_start'